# Fine-tuning the Google-trained models on the 20 BSP training images

Each of the six models trained on the crowdsourced Google Earth dataset is **loaded from its checkpoint and fine-tuned on all 20 Bengaluru training images**. It is early-stopped on the 10 BSP validation images and tested on the 13 BSP test images. Each model is reported **before** (zero-shot) and **after** fine-tuning.

| | |
|---|---|
| Train | 20 BSP training originals (no offline augmentation) |
| Validation | 10 BSP val images: scheduler, early stopping, best-checkpoint choice |
| Test | 13 BSP test images, also per layout (Haralur / HSR / Krystal) |
| Recipe | batch 5, AdamW lr 2e-4, weight decay 1e-2 (none on biases / norms), ReduceLROnPlateau(0.7, patience 7, min 1e-6), max 200 epochs, early stopping patience 10 on val loss, grad clip 1.0, loss 0.6 BCE + 0.4 Dice (+ 0.2 x boundary loss for the two boundary models), best-val-loss weights evaluated |
| Reference | optional: the same ResNet-UNet trained from **ImageNet** weights on the same 20 images with the same recipe, to show what the Google checkpoint adds (`RUN_IMAGENET_REFERENCE`; needs Internet ON) |

One run per model, no fixed seed. Each run takes a few minutes on a GPU.

**Kaggle inputs:** `models` (the six `.pth`) and `bangaloresolarpanels`.

In [ ]:
import os, re, gc, copy, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}', torch.cuda.get_device_name(0) if device.type == 'cuda' else '')

## 1. Model definitions and losses (identical to the training notebooks)

In [ ]:
# Model definitions copied verbatim from the Google training notebooks
# (CD_UnetResnet.ipynb and CD_ResnetUnet_{AxialTransformer,BoundaryHead,BoundaryGuidedAttention,StyleGate}.ipynb)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout_rate=0.15, use_dropout=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if use_dropout:
            layers.append(nn.Dropout2d(dropout_rate))

        layers.extend([
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ])
        if use_dropout:
            layers.append(nn.Dropout2d(dropout_rate))

        self.double_conv = nn.Sequential(*layers)

    def forward(self, x):
        return self.double_conv(x)

# (constructor, ImageNet weights enum name, channels of [stem, layer1, layer2, layer3, layer4])
RESNET_CFG = {
    'resnet18':  (models.resnet18,  'ResNet18_Weights',  [64, 64, 128, 256, 512]),
    'resnet34':  (models.resnet34,  'ResNet34_Weights',  [64, 64, 128, 256, 512]),
    'resnet50':  (models.resnet50,  'ResNet50_Weights',  [64, 256, 512, 1024, 2048]),
    'resnet101': (models.resnet101, 'ResNet101_Weights', [64, 256, 512, 1024, 2048]),
}

class ResNetUNet(nn.Module):
    """U-Net with a ResNet encoder.

    Encoder (strides relative to the input):
        stem   (conv7x7 + BN + ReLU)  /2
        layer1 (after maxpool)        /4
        layer2                        /8
        layer3                        /16
        layer4 (bottleneck)           /32
    Decoder: ConvTranspose2d x2 + skip concat + DoubleConv, five times, back to full resolution.
    Input height/width must be divisible by 32 (256x256 is fine).
    """
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1,
                 dropout_rates=None):
        super().__init__()
        if backbone not in RESNET_CFG:
            raise ValueError(f"backbone must be one of {list(RESNET_CFG)}")

        if dropout_rates is None:
            dropout_rates = {
                'encoder': {'layer3': 0.15, 'layer4': 0.2},   # same idea as U-Net: enc4 = 0.15, bottleneck = 0.2
                'decoder': 0.15,
            }
        self.dropout_rates = dropout_rates

        ctor, weights_name, ch = RESNET_CFG[backbone]
        weights = getattr(models, weights_name).IMAGENET1K_V1 if pretrained else None
        backbone_net = ctor(weights=weights)

        # Optional: support in_ch != 3 by rebuilding the first conv
        if in_ch != 3:
            backbone_net.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Encoder
        self.stem = nn.Sequential(backbone_net.conv1, backbone_net.bn1, backbone_net.relu)  # /2
        self.pool = backbone_net.maxpool                                                    # /4
        self.layer1 = backbone_net.layer1
        self.layer2 = backbone_net.layer2
        self.layer3 = backbone_net.layer3
        self.layer4 = backbone_net.layer4

        self.drop3 = nn.Dropout2d(dropout_rates['encoder']['layer3'])
        self.drop4 = nn.Dropout2d(dropout_rates['encoder']['layer4'])

        c0, c1, c2, c3, c4 = ch
        d = dropout_rates['decoder']

        # Decoder
        self.up4 = nn.ConvTranspose2d(c4, c3, kernel_size=2, stride=2)       # /32 -> /16
        self.dec4 = DoubleConv(c3 * 2, c3, d, use_dropout=True)
        self.up3 = nn.ConvTranspose2d(c3, c2, kernel_size=2, stride=2)       # /16 -> /8
        self.dec3 = DoubleConv(c2 * 2, c2, d, use_dropout=True)
        self.up2 = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)       # /8  -> /4
        self.dec2 = DoubleConv(c1 * 2, c1, d, use_dropout=True)
        self.up1 = nn.ConvTranspose2d(c1, c0, kernel_size=2, stride=2)       # /4  -> /2
        self.dec1 = DoubleConv(c0 * 2, c0, d, use_dropout=True)
        self.up0 = nn.ConvTranspose2d(c0, 32, kernel_size=2, stride=2)       # /2  -> full res
        self.dec0 = DoubleConv(32, 32, d, use_dropout=True)

        self.final_conv = nn.Conv2d(32, out_ch, kernel_size=1)

    def forward(self, x):
        # Encoder
        e0 = self.stem(x)                       # /2
        e1 = self.layer1(self.pool(e0))         # /4
        e2 = self.layer2(e1)                    # /8
        e3 = self.drop3(self.layer3(e2))        # /16
        b = self.drop4(self.layer4(e3))         # /32 (bottleneck)

        # Decoder
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))

        return self.final_conv(d0)

# =====================================================================
# Axial attention (same module as BSP_augmented/Final/Resnet_Axial.ipynb)
# =====================================================================
class AxialAttention2d(nn.Module):
    """Axial self-attention: attend along W (rows), then along H (columns).
    Cost O(HW(H+W)) instead of O((HW)^2). Residual with zero-initialised gate, so at
    init the block is identity and the pretrained ResNet-UNet behaviour is preserved."""
    def __init__(self, dim, heads=None, attn_drop=0.0):
        super().__init__()
        self.heads = heads if heads is not None else max(1, dim // 32)
        assert dim % self.heads == 0
        self.pos = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)       # size-agnostic positional encoding
        self.norm = nn.GroupNorm(1, dim)
        self.qkv_w = nn.Conv2d(dim, dim * 3, 1, bias=False)
        self.qkv_h = nn.Conv2d(dim, dim * 3, 1, bias=False)
        self.proj_w = nn.Conv2d(dim, dim, 1)
        self.proj_h = nn.Conv2d(dim, dim, 1)
        self.attn_drop = attn_drop
        self.gamma = nn.Parameter(torch.zeros(1))

    def _attend(self, x, qkv_layer, proj_layer, axis):
        B, C, H, W = x.shape
        q, k, v = qkv_layer(x).chunk(3, dim=1)
        h = self.heads
        def to_seq(t):                                                  # -> (B*other, heads, L, C/heads)
            if axis == 'w':                                             # sequences run along W
                t = t.permute(0, 2, 3, 1).reshape(B * H, W, h, C // h)
            else:                                                       # sequences run along H
                t = t.permute(0, 3, 2, 1).reshape(B * W, H, h, C // h)
            return t.transpose(1, 2)
        out = F.scaled_dot_product_attention(
            to_seq(q), to_seq(k), to_seq(v),
            dropout_p=self.attn_drop if self.training else 0.0)
        out = out.transpose(1, 2)
        if axis == 'w':
            out = out.reshape(B, H, W, C).permute(0, 3, 1, 2)
        else:
            out = out.reshape(B, W, H, C).permute(0, 3, 2, 1)
        return proj_layer(out.contiguous())

    def forward(self, x):
        y = self.norm(x + self.pos(x))
        y = y + self._attend(y, self.qkv_w, self.proj_w, 'w')          # row attention
        y = y + self._attend(self.norm(y), self.qkv_h, self.proj_h, 'h')  # column attention
        return x + self.gamma * y


class AxialResNetUNet(ResNetUNet):
    """ResNet-UNet with axial attention on the /4, /8 and /16 skip connections."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        _, c1, c2, c3, _ = RESNET_CFG[backbone][2]
        self.ax1, self.ax2, self.ax3 = AxialAttention2d(c1), AxialAttention2d(c2), AxialAttention2d(c3)

    def _enc(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.drop3(self.layer3(e2))
        b = self.drop4(self.layer4(e3))
        return e0, e1, e2, e3, b

    def forward(self, x):
        e0, e1, e2, e3, b = self._enc(x)
        e1, e2, e3 = self.ax1(e1), self.ax2(e2), self.ax3(e3)
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        return self.final_conv(self.dec0(self.up0(d1)))


# =====================================================================
# Transformer on the bottleneck (same module as BSP_augmented/Final/Resnet_AxialTransformer.ipynb)
# =====================================================================
class BottleneckTransformer(nn.Module):
    """Transformer encoder on the /32 bottleneck tokens (8x8 = 64 tokens at 256x256).
    Projects to a small width first so it stays cheap for any ResNet backbone."""
    def __init__(self, in_ch, dim=256, depth=2, heads=8, dropout=0.1):
        super().__init__()
        self.proj_in = nn.Conv2d(in_ch, dim, 1)
        self.pos = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)       # conditional positional encoding
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth, enable_nested_tensor=False)
        self.proj_out = nn.Conv2d(dim, in_ch, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, _, H, W = x.shape
        t = self.proj_in(x)
        t = t + self.pos(t)
        t = self.encoder(t.flatten(2).transpose(1, 2))                  # (B, HW, dim)
        t = t.transpose(1, 2).reshape(B, -1, H, W)
        return x + self.gamma * self.proj_out(t)


class AxialTransResNetUNet(AxialResNetUNet):
    """Axial attention on the /4, /8, /16 skips + 2-layer Transformer on the /32 bottleneck."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1,
                 dropout_rates=None, tr_dim=256, tr_depth=2, tr_heads=8):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        c4 = RESNET_CFG[backbone][2][4]
        self.bottleneck_tr = BottleneckTransformer(c4, tr_dim, tr_depth, tr_heads)

    def forward(self, x):
        e0, e1, e2, e3, b = self._enc(x)
        b = self.bottleneck_tr(b)
        e1, e2, e3 = self.ax1(e1), self.ax2(e2), self.ax3(e3)
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        return self.final_conv(self.dec0(self.up0(d1)))

# =====================================================================
# Auxiliary boundary head (same head as Final Model/ResNet34_BoundaryHead.ipynb)
# =====================================================================
class BoundaryHeadResNetUNet(ResNetUNet):
    """ResNet-UNet + a small boundary head on the full-resolution decoder features.
    The segmentation path is unchanged; the head only adds a training signal.
    forward() returns (segmentation_logits, boundary_logits)."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        self.boundary_head = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, out_ch, kernel_size=1),
        )

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.drop3(self.layer3(e2))
        b = self.drop4(self.layer4(e3))
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))
        return self.final_conv(d0), self.boundary_head(d0)

# =====================================================================
# Boundary-guided attention (same module as Final Model/resnet34_boundaryguidedattention.ipynb)
# =====================================================================
class BoundaryGuidedResNetUNet(ResNetUNet):
    """ResNet-UNet + boundary head whose prediction reweights the full-resolution decoder
    features before the segmentation head:  d0 + scale * d0 * sigmoid(boundary).
    scale starts at 0, so training starts from the plain ResNet-UNet.
    forward() returns (segmentation_logits, boundary_logits)."""
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        self.boundary_feature = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )
        self.boundary_head = nn.Conv2d(16, out_ch, kernel_size=1)
        self.boundary_attention_scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.layer1(self.pool(e0))
        e2 = self.layer2(e1)
        e3 = self.drop3(self.layer3(e2))
        b = self.drop4(self.layer4(e3))
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        d0 = self.dec0(self.up0(d1))

        boundary_logits = self.boundary_head(self.boundary_feature(d0))
        boundary_attention = torch.sigmoid(boundary_logits)
        refined = d0 + self.boundary_attention_scale * (d0 * boundary_attention)
        return self.final_conv(refined), boundary_logits

# =====================================================================
# StyleGate (same module as Final Model/UsingAttentions/Attentions.ipynb)
# =====================================================================
class StyleGate(nn.Module):
    '''Style-invariant channel gate:
        out = x + gamma * g * (IN(x) - x),   g = sigmoid(MLP([channel mean, channel std]))  (per channel)
    IN(x) is the instance-normalised feature: the per-image colour / brightness / contrast statistics
    are removed, which is mostly what changes between regions (Google Earth vs Bengaluru).
    g lets every channel decide how much of the style-free version it wants.
    gamma starts at 0 -> exact identity at the start of training; the network learns whether to use it.'''
    def __init__(self, ch, reduction=4):
        super().__init__()
        self.inorm = nn.InstanceNorm2d(ch, affine=False)
        hid = max(ch // reduction, 8)
        self.mlp = nn.Sequential(nn.Linear(2 * ch, hid), nn.ReLU(inplace=True), nn.Linear(hid, ch))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        mu = x.mean(dim=(2, 3))
        sd = x.var(dim=(2, 3), unbiased=False).add(1e-5).sqrt()
        g = torch.sigmoid(self.mlp(torch.cat([mu, sd], dim=1)))[:, :, None, None]
        return x + self.gamma * g * (self.inorm(x) - x)


class StyleGateResNetUNet(ResNetUNet):
    '''ResNet-UNet with a StyleGate after the stem (/2) and after layer1 (/4),
    i.e. where low-level appearance (colour, texture, contrast) is encoded.'''
    def __init__(self, backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=None):
        super().__init__(backbone, pretrained, in_ch, out_ch, dropout_rates)
        c0, c1, _, _, _ = RESNET_CFG[backbone][2]
        self.style0 = StyleGate(c0)
        self.style1 = StyleGate(c1)

    def forward(self, x):
        e0 = self.style0(self.stem(x))                  # /2
        e1 = self.style1(self.layer1(self.pool(e0)))    # /4
        e2 = self.layer2(e1)                            # /8
        e3 = self.drop3(self.layer3(e2))                # /16
        b = self.drop4(self.layer4(e3))                 # /32
        d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e0], dim=1))
        return self.final_conv(self.dec0(self.up0(d1)))

In [ ]:
# Losses copied verbatim from the training notebooks:
#   dice_coef / DiceLoss / combined_loss (0.6 BCE + 0.4 Dice) -> every notebook
#   BOUNDARY_WEIGHT / boundary_target_from_mask / boundary_loss_fn -> boundary notebooks

def dice_coef(pred, target, smooth=1e-6):
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=(2,3))
    denom = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    dice = (2. * intersection + smooth) / (denom + smooth)
    return dice.mean()


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        dice = dice_coef(pred, target, smooth=self.smooth)
        return 1.0 - dice


def combined_loss(logits, mask, bce_weight=0.6, dice_weight=0.4):
    bce_loss = nn.BCEWithLogitsLoss()
    bce = bce_loss(logits, mask)
    probs = torch.sigmoid(logits)
    dice = DiceLoss()(probs, mask)
    return bce_weight * bce + dice_weight * dice


BOUNDARY_WEIGHT = 0.2


def boundary_target_from_mask(mask):
    """~2 px wide boundary band (dilation minus erosion with a 3x3 window) from a binary mask."""
    dilated = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
    eroded = 1.0 - F.max_pool2d(1.0 - mask, kernel_size=3, stride=1, padding=1)
    return (dilated - eroded).clamp(0.0, 1.0)


def boundary_loss_fn(boundary_logits, mask):
    """0.5 * class-balanced BCE + 0.5 * Dice on the boundary band."""
    target = boundary_target_from_mask(mask)
    positives = target.sum()
    negatives = target.numel() - positives
    pos_weight = (negatives / positives.clamp_min(1.0)).clamp(min=1.0, max=20.0).reshape(1)
    bce = F.binary_cross_entropy_with_logits(boundary_logits, target, pos_weight=pos_weight)
    return 0.5 * bce + 0.5 * DiceLoss()(torch.sigmoid(boundary_logits), target)

## 2. Configuration

In [ ]:
BSP_ROOT = '/kaggle/input/datasets/dewangchoudhary/bangaloresolarpanels/BangaloreSolarPanels'
CKPT_DIR = '/kaggle/input/datasets/dewangchoudhary/models/Models'
OUT_DIR = '/kaggle/working/finetune20'
os.makedirs(OUT_DIR, exist_ok=True)

MODELS_TO_RUN = ['ResNet-UNet (baseline)', 'Axial', 'Axial + Transformer',
                 'Boundary Head', 'Boundary-Guided Attention', 'StyleGate']
RUN_IMAGENET_REFERENCE = True      # also train ResNet-UNet from ImageNet weights on the same 20 images (needs Internet ON)
SAVE_CKPTS = True                  # save each fine-tuned model (~100 MB each)

CFG = dict(batch_size=5, lr=2e-4, weight_decay=1e-2, max_epochs=200, patience=10, min_delta=1e-4,
           sched_factor=0.7, sched_patience=7, min_lr=1e-6, grad_clip=1.0)
IMG_SIZE = (256, 256)
THRESH = 0.5
DROPOUT_CONFIG = {'encoder': {'layer3': 0.15, 'layer4': 0.2}, 'decoder': 0.15}

# (class, Google checkpoint, has boundary output)
MODEL_SPECS = {
    'ResNet-UNet (baseline)':    (ResNetUNet,               'CD_ResnetUnet.pth',                         False),
    'Axial':                     (AxialResNetUNet,          'CD_ResnetUnet_Axial.pth',                   False),
    'Axial + Transformer':       (AxialTransResNetUNet,     'CD_ResnetUnet_AxialTransformer.pth',        False),
    'Boundary Head':             (BoundaryHeadResNetUNet,   'CD_ResnetUnet_BoundaryHead.pth',            True),
    'Boundary-Guided Attention': (BoundaryGuidedResNetUNet, 'CD_ResnetUnet_BoundaryGuidedAttention.pth', True),
    'StyleGate':                 (StyleGateResNetUNet,      'CD_ResnetUnet_StyleGate.pth',               False),
}
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')


def ci_sub(parent, name):
    if parent is None or not os.path.isdir(parent):
        return None
    for d in os.listdir(parent):
        if d.lower() == name.lower() and os.path.isdir(os.path.join(parent, d)):
            return os.path.join(parent, d)
    return None


def split_dirs(root):
    out = {}
    for s in ('train', 'val', 'test'):
        sd = ci_sub(root, s)
        im, mk = ci_sub(sd, 'img'), ci_sub(sd, 'masks')
        if im is None or mk is None:
            return None
        out[s] = (im, mk)
    return out


# fall back to searching /kaggle/input if the default paths are not there
if not split_dirs(BSP_ROOT) or not os.path.isdir(CKPT_DIR):
    print('Default paths not found - searching /kaggle/input ...')
    for dp, _, fns in os.walk('/kaggle/input', followlinks=True):
        sd = split_dirs(dp)
        if not split_dirs(BSP_ROOT) and sd and 'augment' not in dp.lower():
            BSP_ROOT = dp
        if not os.path.isdir(CKPT_DIR) and 'CD_ResnetUnet.pth' in fns:
            CKPT_DIR = dp
print('BSP_ROOT =', BSP_ROOT)
print('CKPT_DIR =', CKPT_DIR)
for m in MODELS_TO_RUN:
    p = os.path.join(CKPT_DIR, MODEL_SPECS[m][1])
    print(f'  {m:27s} {"OK" if os.path.isfile(p) else "MISSING (skipped)"}  {p}')

## 3. Load BSP (train 20 / val 10 / test 13)
Same preprocessing as all training notebooks: resize to 256x256, ImageNet normalisation, mask = pixel > 0.

In [ ]:
NORM = transforms.Compose([transforms.ToTensor(),
                           transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])


def layout_of(stem):
    s = stem.lower()
    if s.startswith('behind_layout') or s.startswith('haralur'):
        return 'Haralur'
    if s.startswith('hsr'):
        return 'HSR'
    if s.startswith('kristal') or s.startswith('krystal'):
        return 'Krystal'
    return 'Other'


def load_split(split):
    img_dir, mask_dir = split_dirs(BSP_ROOT)[split]
    masks = {}
    for f in os.listdir(mask_dir):
        if f.lower().endswith(IMG_EXTS):
            st = os.path.splitext(f)[0]
            masks[st[:-5] if st.lower().endswith('_mask') else st] = f
    recs = []
    for f in sorted(os.listdir(img_dir)):
        st = os.path.splitext(f)[0]
        if not f.lower().endswith(IMG_EXTS) or st not in masks:
            continue
        img = Image.open(os.path.join(img_dir, f)).convert('RGB').resize(IMG_SIZE)
        lab = Image.open(os.path.join(mask_dir, masks[st])).convert('L').resize(IMG_SIZE, Image.NEAREST)
        recs.append(dict(name=st, layout=layout_of(st), rgb=np.array(img), x=NORM(img),
                         y=torch.from_numpy((np.array(lab) > 0).astype(np.float32))))
    X = torch.stack([r['x'] for r in recs])
    Y = torch.stack([r['y'] for r in recs]).unsqueeze(1)
    meta = pd.DataFrame([{k: r[k] for k in ('name', 'layout')} for r in recs])
    return X, Y, meta, np.stack([r['rgb'] for r in recs])


X_tr, Y_tr, meta_tr, _ = load_split('train')
X_val, Y_val, meta_val, _ = load_split('val')
X_test, Y_test, meta_test, RGB_test = load_split('test')
print(f'train {len(X_tr)} | val {len(X_val)} | test {len(X_test)}')
display(pd.DataFrame({s: m.layout.value_counts() for s, m in (('train', meta_tr), ('val', meta_val), ('test', meta_test))}).fillna(0).astype(int))
TEST_LAYOUT_IDX = {lay: np.where(meta_test.layout == lay)[0] for lay in ('Haralur', 'HSR', 'Krystal')}

## 4. Helpers

In [ ]:
def make_model(name, init='Google'):
    cls, ckpt, _ = MODEL_SPECS[name]
    if init == 'Google':
        m = cls(backbone='resnet34', pretrained=False, in_ch=3, out_ch=1, dropout_rates=DROPOUT_CONFIG)
        sd = torch.load(os.path.join(CKPT_DIR, ckpt), map_location='cpu')
        m.load_state_dict({(k[7:] if k.startswith('module.') else k): v for k, v in sd.items()}, strict=True)
    else:   # 'ImageNet'
        m = cls(backbone='resnet34', pretrained=True, in_ch=3, out_ch=1, dropout_rates=DROPOUT_CONFIG)
    return m.to(device)


def seg_out(out):
    return out[0] if isinstance(out, (tuple, list)) else out


@torch.no_grad()
def predict(model, X, bs=8):
    model.eval()
    return torch.cat([torch.sigmoid(seg_out(model(X[i:i + bs].to(device)))).cpu() for i in range(0, len(X), bs)])


@torch.no_grad()
def seg_loss(model, X, Y, bs=2):
    # segmentation loss only: used for the scheduler, early stopping and model choice (as in all notebooks)
    model.eval()
    tot = 0.0
    for i in range(0, len(X), bs):
        xb, yb = X[i:i + bs].to(device), Y[i:i + bs].to(device)
        tot += combined_loss(seg_out(model(xb)), yb).item() * len(xb)
    return tot / len(X)


def metrics(P, Y, idx=None, thr=THRESH):
    if idx is not None:
        P, Y = P[idx], Y[idx]
    pred, gt = P >= thr, Y > 0.5
    tp = (pred & gt).sum().item(); fp = (pred & ~gt).sum().item()
    fn = (~pred & gt).sum().item(); tn = (~pred & ~gt).sum().item()
    eps = 1e-8
    iou = tp / (tp + fp + fn + eps)
    prec, rec = tp / (tp + fp + eps), tp / (tp + fn + eps)
    return {'IoU': iou, 'mIoU': (iou + tn / (tn + fp + fn + eps)) / 2, 'Dice': 2 * tp / (2 * tp + fp + fn + eps),
            'Precision': prec, 'Recall': rec, 'F1': 2 * prec * rec / (prec + rec + eps),
            'Accuracy': (tp + tn) / (tp + tn + fp + fn + eps)}


def evaluate(model):
    P = predict(model, X_test)
    row = metrics(P, Y_test)
    row['Loss'] = seg_loss(model, X_test, Y_test, bs=1)
    for lay, idx in TEST_LAYOUT_IDX.items():
        row[f'IoU {lay}'] = metrics(P, Y_test, idx)['IoU']
    row['val IoU'] = metrics(predict(model, X_val), Y_val)['IoU']
    return row, P


def fine_tune(model, has_boundary):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (no_decay if (p.ndim == 1 or n.endswith('.bias')) else decay).append(p)
    opt = optim.AdamW([{'params': decay, 'weight_decay': CFG['weight_decay']},
                       {'params': no_decay, 'weight_decay': 0.0}], lr=CFG['lr'])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=CFG['sched_factor'],
                                                 patience=CFG['sched_patience'], min_lr=CFG['min_lr'])
    loader = DataLoader(TensorDataset(X_tr, Y_tr), batch_size=CFG['batch_size'], shuffle=True, num_workers=0)

    hist = {'train_loss': [], 'val_loss': [], 'val_iou': []}
    best, best_state, best_epoch, counter = None, None, 0, 0
    for epoch in range(1, CFG['max_epochs'] + 1):
        model.train()
        tot = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            if has_boundary:
                loss = combined_loss(out[0], yb) + BOUNDARY_WEIGHT * boundary_loss_fn(out[1], yb)
            else:
                loss = combined_loss(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG['grad_clip'])
            opt.step()
            tot += loss.item() * len(xb)
        vl = seg_loss(model, X_val, Y_val)
        sched.step(vl)
        hist['train_loss'].append(tot / len(X_tr)); hist['val_loss'].append(vl)
        hist['val_iou'].append(metrics(predict(model, X_val), Y_val)['IoU'])
        if best is None or best - vl > CFG['min_delta']:
            best, best_state, best_epoch, counter = vl, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            counter += 1
            if counter >= CFG['patience']:
                break
    model.load_state_dict(best_state)          # best-val-loss weights
    return model, hist, epoch, best_epoch

## 5. Fine-tune every model
For each model: evaluate the Google checkpoint (**before**), fine-tune on the 20 training images, then evaluate the best-val-loss weights (**after**).

In [ ]:
rows, hists, P_before, P_after = [], {}, {}, {}
jobs = [(m, 'Google') for m in MODELS_TO_RUN if os.path.isfile(os.path.join(CKPT_DIR, MODEL_SPECS[m][1]))]
if RUN_IMAGENET_REFERENCE:
    jobs.append(('ResNet-UNet (baseline)', 'ImageNet'))

for name, init in jobs:
    label = name if init == 'Google' else f'{name} [ImageNet init]'
    t0 = time.time()
    model = make_model(name, init)
    if init == 'Google':
        before, P_before[label] = evaluate(model)
        rows.append({'Model': label, 'Stage': 'before (zero-shot)', **before})
    model, hists[label], n_ep, best_ep = fine_tune(model, MODEL_SPECS[name][2])
    after, P_after[label] = evaluate(model)
    rows.append({'Model': label, 'Stage': 'after fine-tuning', 'epochs': n_ep, 'best epoch': best_ep, **after})
    if SAVE_CKPTS:
        torch.save(model.state_dict(), f"{OUT_DIR}/FT20_{MODEL_SPECS[name][1].replace('.pth', '')}{'' if init == 'Google' else '_ImageNetInit'}.pth")
    print(f"{label:40s} | epochs {n_ep:3d} (best {best_ep:3d}) | "
          + (f"test IoU {before['IoU']:.4f} -> {after['IoU']:.4f}" if init == 'Google' else f"test IoU {after['IoU']:.4f}")
          + f" | F1 {after['F1']:.4f} | {(time.time() - t0) / 60:.1f} min")
    del model
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results.to_csv(f'{OUT_DIR}/finetune20_results.csv', index=False)

## 6. Results on the 13 BSP test images

In [ ]:
after = results[results.Stage == 'after fine-tuning'].set_index('Model')
before = results[results.Stage == 'before (zero-shot)'].set_index('Model')

print('After fine-tuning on the 20 BSP training images')
display(after[['IoU', 'mIoU', 'Dice', 'Precision', 'Recall', 'F1', 'Accuracy', 'Loss', 'epochs', 'best epoch']])

print('Before vs after (test IoU / F1)')
cmp = pd.DataFrame({'IoU before': before['IoU'], 'IoU after': after['IoU'],
                    'F1 before': before['F1'], 'F1 after': after['F1']}).reindex(after.index)
cmp['IoU gain'] = cmp['IoU after'] - cmp['IoU before']
display(cmp)

print('Per-layout test IoU after fine-tuning')
display(after[['IoU Haralur', 'IoU HSR', 'IoU Krystal', 'val IoU']])

fig, ax = plt.subplots(figsize=(11, 4.5))
xs = np.arange(len(cmp))
b1 = ax.bar(xs - 0.2, cmp['IoU before'].fillna(0), 0.4, label='before (zero-shot)')
b2 = ax.bar(xs + 0.2, cmp['IoU after'], 0.4, label='after fine-tuning on 20 BSP images')
ax.bar_label(b1, fmt='%.3f', fontsize=8); ax.bar_label(b2, fmt='%.3f', fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels(cmp.index, rotation=15, ha='right')
ax.set_ylabel('test IoU (13 BSP test images)'); ax.set_ylim(0, 1); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/before_after_iou.png', dpi=130, bbox_inches='tight'); plt.show()

## 7. Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for label, h in hists.items():
    ep = np.arange(1, len(h['train_loss']) + 1)
    axes[0].plot(ep, h['train_loss'], label=label)
    axes[1].plot(ep, h['val_loss'], label=label)
    axes[2].plot(ep, h['val_iou'], label=label)
for a, t in zip(axes, ('train loss', 'val loss (segmentation)', 'val IoU')):
    a.set_title(t); a.set_xlabel('epoch'); a.grid(alpha=0.3)
axes[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=130, bbox_inches='tight'); plt.show()

## 8. Test predictions
**green = correct panel pixels**, **red = false positive**, **blue = missed panel**.

In [ ]:
def overlay(rgb, pred, gt):
    out = rgb.astype(np.float32) / 255.0 * 0.55
    for msk, c in ((pred & gt, (0.1, 0.95, 0.1)), (pred & ~gt, (1.0, 0.15, 0.15)), (~pred & gt, (0.2, 0.45, 1.0))):
        out[msk] = 0.35 * out[msk] + 0.65 * np.array(c)
    return np.clip(out, 0, 1)


def show(P_dict, title, fname):
    cols = 2 + len(P_dict)
    fig, axes = plt.subplots(len(RGB_test), cols, figsize=(2.4 * cols, 2.5 * len(RGB_test)), squeeze=False)
    for r in range(len(RGB_test)):
        gt = Y_test[r, 0].numpy() > 0.5
        axes[r, 0].imshow(RGB_test[r]); axes[r, 0].set_title(meta_test.name.iloc[r], fontsize=7)
        axes[r, 1].imshow(gt, cmap='gray'); axes[r, 1].set_title('ground truth', fontsize=8)
        for c, (label, P) in enumerate(P_dict.items()):
            pred = P[r, 0].numpy() >= THRESH
            iou = (pred & gt).sum() / max((pred | gt).sum(), 1)
            axes[r, c + 2].imshow(overlay(RGB_test[r], pred, gt))
            axes[r, c + 2].set_title(f'{label}\nIoU {iou:.2f}', fontsize=7)
        for a in axes[r]:
            a.axis('off')
    fig.suptitle(title, fontsize=12, y=1.0)
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{fname}', dpi=100, bbox_inches='tight'); plt.show()


show(P_after, 'After fine-tuning on the 20 BSP training images', 'overlay_after_finetuning.png')
BASELINE_NAME = 'ResNet-UNet (baseline)'
if BASELINE_NAME in P_before:
    show({'baseline before': P_before[BASELINE_NAME], 'baseline after': P_after[BASELINE_NAME]},
         'ResNet-UNet: zero-shot vs fine-tuned', 'overlay_baseline_before_after.png')

## 9. Saved outputs

In [ ]:
for f in sorted(os.listdir(OUT_DIR)):
    print(f'{OUT_DIR}/{f}  ({os.path.getsize(os.path.join(OUT_DIR, f)) / 1024 / 1024:.1f} MB)')